# Sailor2-8B classification baseline on Kaggle

Enable Internet and a CUDA GPU; attach private `VGEC 2026_v1.json` and `VGEC_classifier.json` files matching the local baseline. This notebook clones `kaggle-linux` and trains **only the baseline classifier**: no OOF hints, no correction model, no Task 1 to Task 2 transfer.

`configs/baseline.json` retains pinned Sailor2-8B-Chat, NF4 QLoRA, rank 16 / alpha 32 / dropout 0.05, seed 42, five epochs, LR 5e-5 and effective batch eight. The unchanged audited data has 504 training rows (312 retained originals + 192 synthetic rows) and 81 original validation rows. A fresh fit selects its own best epoch by validation macro F1; it does not reuse the local trained adapter or guarantee identical GPU numerics.

Inputs are discovered under `/kaggle/input`; set explicit paths below if several attachments contain the same filename. Outputs use `/kaggle/output`, linked to `/kaggle/working/output` for notebook-version retention. Keep all dataset and checkpoint outputs private. Publish `kaggle-linux` before cloning it from Kaggle.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import signal
import sys

# Use PyTorch; avoid importing optional TensorFlow/Flax stacks.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")

REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "kaggle-linux"
PROJECT_SUBDIR = "."  # The project is at repository root.
CLASSIFICATION_CONFIG = "configs/baseline.json"
CLASSIFICATION_OVERRIDES = {}  # Optional baseline experiment overrides.
ORIGINAL_DATA = None  # e.g. "/kaggle/input/my-private-dataset/VGEC 2026_v1.json"
CLASSIFIER_DATA = None  # e.g. "/kaggle/input/my-private-dataset/VGEC_classifier.json"
USE_HF_SECRET = False
CLONE_DIR = Path("/kaggle/working/gecc-baseline-source")
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
print("Branch:", BRANCH, "| config:", CLASSIFICATION_CONFIG)



In [ ]:
# This cell can be rerun after a kernel restart. Git runs as a subprocess.
# Reuse only a clean checkout of the same repository and branch.
from urllib.parse import urlsplit
if urlsplit(REPO_URL).username is not None:
    raise ValueError("Do not embed GitHub credentials in REPO_URL.")
subprocess.run(["git", "check-ref-format", "--branch", BRANCH], check=True, capture_output=True)
if CLONE_DIR.exists():
    if not (CLONE_DIR / ".git").is_dir():
        raise RuntimeError(f"{CLONE_DIR} exists but is not a Git clone; choose another CLONE_DIR.")
    existing_url = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=CLONE_DIR, text=True).strip()
    if existing_url != REPO_URL:
        raise RuntimeError("CLONE_DIR belongs to a different repository; choose another CLONE_DIR.")
    existing_branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=CLONE_DIR, text=True).strip()
    if existing_branch != BRANCH:
        raise RuntimeError("CLONE_DIR uses a different branch; choose another CLONE_DIR.")
    changes = subprocess.check_output(["git", "status", "--porcelain"], cwd=CLONE_DIR, text=True)
    if changes.strip():
        raise RuntimeError("CLONE_DIR has local changes; preserve them and choose another CLONE_DIR.")

remote = subprocess.run(["git", "ls-remote", "--exit-code", "--heads", "--", REPO_URL,
                         f"refs/heads/{BRANCH}"], capture_output=True, text=True)
if remote.returncode == 2:
    raise RuntimeError(f"Branch {BRANCH!r} does not exist on GitHub. Push it or select a published BRANCH.")
if remote.returncode != 0:
    raise RuntimeError("Cannot read the Git repository. Enable Kaggle Internet and check repository access.")
if CLONE_DIR.exists():
    subprocess.run(["git", "fetch", "origin", f"refs/heads/{BRANCH}"], cwd=CLONE_DIR, check=True)
    subprocess.run(["git", "merge", "--ff-only", "FETCH_HEAD"], cwd=CLONE_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
                    "--", REPO_URL, str(CLONE_DIR)], check=True)
PROJECT_DIR = (CLONE_DIR / PROJECT_SUBDIR).resolve()
if not PROJECT_DIR.is_relative_to(CLONE_DIR.resolve()):
    raise ValueError("PROJECT_SUBDIR must stay inside the cloned repository.")
for filename in ["requirements-kaggle.txt", "model/classification.py", CLASSIFICATION_CONFIG]:
    if not (PROJECT_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing {filename} in {PROJECT_DIR}; check the selected branch and PROJECT_SUBDIR.")
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH, "project_subdir": PROJECT_SUBDIR,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CLONE_DIR, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))



In [ ]:
# Keep Kaggle's CUDA-enabled torch; install the tested non-torch stack.
# GPU precision is resolved by placement_preflight in the next cell.
from importlib.metadata import version
import tempfile

installed_torch = version("torch")
packages = ["transformers", "peft", "accelerate", "safetensors", "huggingface-hub", "tokenizers", "bitsandbytes"]
imported_before = {name: version(name) for name in packages if name.replace("-", "_") in sys.modules}
with tempfile.TemporaryDirectory() as temporary:
    constraints = Path(temporary) / "constraints.txt"
    constraints.write_text(f"torch=={installed_torch}\n", encoding="utf-8")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements-kaggle.txt"),
                    "--constraint", str(constraints)], cwd=PROJECT_DIR, check=True)
changed_imports = [name for name, old in imported_before.items() if version(name) != old]
if changed_imports:
    raise RuntimeError(f"Restart the kernel to reload {changed_imports}, then rerun from the parameter cell; the clean clone is reused.")
# Check the training stack directly; unrelated packages in Kaggle's image
# can have pre-existing conflicts that do not affect this PyTorch baseline.
subprocess.run([sys.executable, "-c", "import torch, transformers, peft, accelerate, bitsandbytes; print('Training imports passed')"], check=True)
print("Pinned dependencies installed; retained torch", installed_torch)



In [ ]:
if USE_HF_SECRET:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

def apply_overrides(target, overrides):
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(target.get(key), dict):
            apply_overrides(target[key], value)
        else:
            target[key] = value

config = json.loads((PROJECT_DIR / CLASSIFICATION_CONFIG).read_text(encoding="utf-8"))
apply_overrides(config, CLASSIFICATION_OVERRIDES)
config["data"].update(path=ORIGINAL_DATA, augmentation_path=CLASSIFIER_DATA,
                      input_root="/kaggle/input")
config["run"]["output_root"] = "/kaggle/output"
sys.path.insert(0, str(PROJECT_DIR))
from model.runtime import load_config, resolve_data
from model.data import load_records, load_classifier_split, split_records
from model.classification import placement_preflight

# /kaggle/working is retained by Save Version; expose the requested output path.
PERSISTED_OUTPUT = Path("/kaggle/working/output")
PERSISTED_OUTPUT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = Path("/kaggle/output")
if not OUTPUT_ROOT.exists() and not OUTPUT_ROOT.is_symlink():
    OUTPUT_ROOT.symlink_to(PERSISTED_OUTPUT, target_is_directory=True)
if OUTPUT_ROOT.resolve() != PERSISTED_OUTPUT.resolve():
    raise RuntimeError("/kaggle/output must link to /kaggle/working/output so Save Version retains artifacts.")
original_path = resolve_data(config)
mixed_config = json.loads(json.dumps(config))
mixed_config["data"].update(path=config["data"]["augmentation_path"],
                           filename=config["data"]["augmentation_filename"])
classifier_path = resolve_data(mixed_config)
for data_path in (original_path, classifier_path):
    if not data_path.is_relative_to(Path("/kaggle/input")):
        raise ValueError(f"Baseline input must be an attached Kaggle dataset: {data_path}")
config["data"].update(path=str(original_path), augmentation_path=str(classifier_path))
CONFIG_PATH = PERSISTED_OUTPUT / f"baseline-config-{RUN_ID}.json"
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
load_config(CONFIG_PATH)
train, provenance = load_classifier_split(original_path, classifier_path,
                                        config["data"]["validation_fraction"], config["run"]["seed"])
_, validation = split_records(load_records(original_path), config["data"]["validation_fraction"], config["run"]["seed"])
print(json.dumps({"original_data": str(original_path), "classifier_data": str(classifier_path),
                  "train_rows": len(train), "validation_rows": len(validation)}, indent=2))
print(json.dumps(placement_preflight(config["model"], config["run"]["device"]), indent=2))
OUTPUT_DIR = OUTPUT_ROOT / f"sailor2-baseline-{RUN_ID}"
command = [sys.executable, "-m", "model.classification",
           "--config", str(CONFIG_PATH), "--output-dir", str(OUTPUT_DIR)]


In [ ]:
# Train only the baseline classifier and reload its saved adapter for evaluation.
LOG_PATH = OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}.log"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
with LOG_PATH.open("w", encoding="utf-8") as log_file:
    log_file.write(json.dumps(SOURCE_INFO, ensure_ascii=False) + "\n")
    log_file.flush()
    with subprocess.Popen(command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1,
                          start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log_file.write(line)
                log_file.flush()
            returncode = process.wait()
        except BaseException:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


In [ ]:
from IPython.display import JSON, FileLink, display
for filename in ["metrics.json", "run_manifest.json"]:
    print(filename)
    display(JSON(json.loads((OUTPUT_DIR / filename).read_text(encoding="utf-8"))))
import shutil
ARCHIVE_PATH = Path(shutil.make_archive(str(PERSISTED_OUTPUT / OUTPUT_DIR.name) + "-artifacts", "zip", OUTPUT_DIR))
display(FileLink(str(ARCHIVE_PATH)))
display(FileLink(str(LOG_PATH.resolve())))
print("Sailor2 baseline classifier:", OUTPUT_DIR / "classifier")
print("Save a private notebook version to retain /kaggle/working/output.")


## Saved baseline use

The run saves its resolved config, train/validation IDs, predictions, metrics, epoch history and source manifest. `classifier/` contains the tokenizer, preprocessing, hierarchical model metadata and the LoRA adapter including the width-five score head. The selected epoch is measured on the same holdout, not an independent test set. The artifact ZIP and log live under `/kaggle/working/output`, also accessible as `/kaggle/output`.

```python
from model.classification import classify
predictions = classify(OUTPUT_DIR / "classifier", records, device="cuda", batch_size=1)
```

For a direct Kaggle terminal run from the cloned repository, install `requirements-kaggle.txt`, create the output link as above, then run:

```bash
python -m model.classification --config configs/baseline.json
```

Only the two classification datasets are required. No balanced correction bundle, OOF hints or correction checkpoint is produced.
